# Diffusion Models: From Noise to Images

**Learning Objectives:**
- Understand the core intuition behind diffusion models
- Learn the forward diffusion process (adding noise)
- Learn the reverse diffusion process (denoising)
- Implement Denoising Diffusion Probabilistic Models (DDPM)
- Understand noise scheduling and sampling strategies
- Build and train a diffusion model on MNIST
- Compare diffusion models to VAEs and GANs

**What We'll Build:**
A complete DDPM implementation that generates MNIST digits by iteratively denoising random noise.

## 1. Introduction: What Are Diffusion Models?

**Diffusion models** are the current state-of-the-art approach for image generation, powering systems like:
- DALL-E 2
- Stable Diffusion
- Midjourney
- Imagen

### The Core Idea

Imagine watching a video in reverse:
1. **Forward process**: A clear image gradually becomes pure noise (like watching ink diffuse in water)
2. **Reverse process**: Learn to reverse this - turn noise back into a clear image

### Why Diffusion Models?

**Advantages over VAE/GAN:**
- **Better image quality**: Sharper than VAEs, more stable than GANs
- **Training stability**: No adversarial training, no mode collapse
- **Theoretical foundation**: Strong probabilistic framework
- **Flexible sampling**: Can trade quality for speed

**The trade-off:**
- **Slower sampling**: Need many denoising steps (50-1000)
- **More compute**: Training and inference are expensive

### Two Perspectives

Diffusion models can be understood from two angles:
1. **Probabilistic**: Learn the reverse of a Markov chain
2. **Denoising**: Train a model to remove noise at various noise levels

## 2. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math

# Shared library utilities
from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
set_seed(42)

# Device setup
device = get_device()
print(f"Using device: {device}")

## 3. Load MNIST Dataset

We'll use MNIST to keep training manageable while learning the concepts.

In [ ]:
# Transform: Convert to tensor and scale to [-1, 1]
# Diffusion models typically work better with zero-centered data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Scale to [-1, 1]
])

# Load datasets
train_dataset = datasets.MNIST(
    root='./tmp/data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./tmp/data',
    train=False,
    download=True,
    transform=transform
)

# Create data loaders
batch_size = 128
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

print(f"Training samples: {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Image shape: {train_dataset[0][0].shape}")

In [ ]:
# Visualize some samples
def show_images(images, title="", nrow=8):
    """Display a grid of images."""
    # Denormalize from [-1, 1] to [0, 1]
    images = (images + 1) / 2
    images = torch.clamp(images, 0, 1)
    
    grid = make_grid(images, nrow=nrow, padding=2)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Show sample images
sample_batch = next(iter(train_loader))[0][:64]
show_images(sample_batch, "Sample MNIST Images")

## 4. The Forward Diffusion Process

### Intuition

The **forward process** gradually adds Gaussian noise to an image over $T$ timesteps:

$$x_0 \rightarrow x_1 \rightarrow x_2 \rightarrow \cdots \rightarrow x_T$$

Where:
- $x_0$ is the original clean image
- $x_T$ is pure noise (approximately $\mathcal{N}(0, I)$)
- Each step adds a small amount of noise

### Mathematical Definition

At each timestep $t$, we add noise according to:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} x_{t-1}, \beta_t I)$$

Where $\beta_t$ is the **noise schedule** - how much noise to add at step $t$.

### The Beautiful Property: Closed Form

We don't need to apply $T$ steps sequentially! We can jump directly to any timestep:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1-\bar{\alpha}_t) I)$$

Where:
- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ (cumulative product)

**In practice:**
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This makes training efficient!

## 5. Noise Scheduling

The **noise schedule** $\{\beta_1, \beta_2, \ldots, \beta_T\}$ controls how quickly we add noise.

### Linear Schedule (Original DDPM)

$$\beta_t = \beta_{\text{start}} + \frac{t}{T}(\beta_{\text{end}} - \beta_{\text{start}})$$

Typical values:
- $\beta_{\text{start}} = 0.0001$
- $\beta_{\text{end}} = 0.02$
- $T = 1000$

### Why This Matters

- **Too fast**: Model can't learn to denoise properly
- **Too slow**: Inefficient, need many steps
- **Just right**: Gradual transition from image to noise

In [ ]:
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """
    Linear schedule from DDPM paper.
    
    Args:
        timesteps: Number of diffusion steps
        beta_start: Starting noise level
        beta_end: Ending noise level
    
    Returns:
        betas: Noise schedule [T]
    """
    return torch.linspace(beta_start, beta_end, timesteps)

def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine schedule from "Improved Denoising Diffusion Probabilistic Models".
    Better preserves small noise levels early on.
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

# Visualize schedules
T = 1000
linear_betas = linear_beta_schedule(T)
cosine_betas = cosine_beta_schedule(T)

# Compute alpha_bar for both
def compute_alpha_bar(betas):
    alphas = 1 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return alphas_cumprod

linear_alpha_bar = compute_alpha_bar(linear_betas)
cosine_alpha_bar = compute_alpha_bar(cosine_betas)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot betas
axes[0].plot(linear_betas.numpy(), label='Linear', alpha=0.7)
axes[0].plot(cosine_betas.numpy(), label='Cosine', alpha=0.7)
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel(r'$\beta_t$')
axes[0].set_title('Noise Schedule: Beta Values')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot alpha_bar (signal retention)
axes[1].plot(linear_alpha_bar.numpy(), label='Linear', alpha=0.7)
axes[1].plot(cosine_alpha_bar.numpy(), label='Cosine', alpha=0.7)
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel(r'$\bar{\alpha}_t$ (signal retention)')
axes[1].set_title('Cumulative Alpha: How Much Signal Remains')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At t=500: linear α̅={linear_alpha_bar[499]:.4f}, cosine α̅={cosine_alpha_bar[499]:.4f}")
print(f"At t=999: linear α̅={linear_alpha_bar[999]:.6f}, cosine α̅={cosine_alpha_bar[999]:.6f}")

### Interpretation

- **$\beta_t$**: How much noise to add at step $t$
- **$\bar{\alpha}_t$**: How much of the original signal remains at step $t$

Notice:
- Linear schedule: Aggressive early, signal decays quickly
- Cosine schedule: Gentler, preserves signal longer

For MNIST (simple images), linear works well. For complex images, cosine is often better.

## 6. Visualizing the Forward Process

Let's see what happens to an image as we add noise.

In [ ]:
def forward_diffusion_sample(x0, t, sqrt_alpha_bar, sqrt_one_minus_alpha_bar):
    """
    Sample x_t from q(x_t | x_0) using the closed form.
    
    Args:
        x0: Original image [B, C, H, W]
        t: Timestep (int or tensor of ints)
        sqrt_alpha_bar: Precomputed sqrt(α̅_t) for all t
        sqrt_one_minus_alpha_bar: Precomputed sqrt(1 - α̅_t) for all t
    
    Returns:
        xt: Noisy image at timestep t
        noise: The noise that was added
    """
    # Sample noise
    noise = torch.randn_like(x0)
    
    # Get coefficients for timestep t
    sqrt_alpha_bar_t = sqrt_alpha_bar[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alpha_bar_t = sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
    
    # Apply closed form: x_t = sqrt(α̅_t) * x_0 + sqrt(1-α̅_t) * ε
    xt = sqrt_alpha_bar_t * x0 + sqrt_one_minus_alpha_bar_t * noise
    
    return xt, noise

In [ ]:
# Setup diffusion parameters
T = 1000
betas = linear_beta_schedule(T)
alphas = 1 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# Precompute values we'll need
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod)

print(f"Diffusion setup: T={T} steps")
print(f"β range: [{betas[0]:.6f}, {betas[-1]:.6f}]")
print(f"α̅ range: [{alphas_cumprod[0]:.6f}, {alphas_cumprod[-1]:.6f}]")

In [ ]:
# Visualize forward process
sample_image = sample_batch[0:1].clone()
timesteps_to_show = [0, 50, 100, 200, 400, 600, 800, 999]

noisy_images = []
for t in timesteps_to_show:
    if t == 0:
        noisy_images.append(sample_image)
    else:
        t_tensor = torch.tensor([t])
        noisy, _ = forward_diffusion_sample(
            sample_image, t_tensor,
            sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod
        )
        noisy_images.append(noisy)

# Display
noisy_images = torch.cat(noisy_images, dim=0)
grid = make_grid(noisy_images, nrow=8, padding=2, normalize=True, value_range=(-1, 1))

plt.figure(figsize=(15, 3))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title(f"Forward Diffusion Process: t = {timesteps_to_show}")
plt.axis('off')
plt.tight_layout()
plt.show()

print("Notice how the image gradually becomes pure noise!")

### Key Observation

By $t \approx 600$, the image is almost completely noise. By $t=999$, it's indistinguishable from random noise.

**Our goal**: Learn to reverse this process - start from noise and recover the original image!

## 7. The Reverse Diffusion Process

### The Challenge

We want to learn the **reverse process**:

$$p_\theta(x_{t-1} | x_t)$$

Starting from noise $x_T \sim \mathcal{N}(0, I)$, iteratively denoise to get $x_0$.

### The Trick: Predict the Noise

Instead of predicting $x_{t-1}$ directly, we predict the **noise** $\epsilon$ that was added!

**Why?** The noise is the same at all timesteps (Gaussian), making it easier to learn.

### The Model

Train a neural network $\epsilon_\theta(x_t, t)$ to predict the noise $\epsilon$ given:
- Noisy image $x_t$
- Timestep $t$ (important! Same image has different noise levels at different $t$)

### Training Objective (Simplified)

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

**Algorithm:**
1. Sample training image $x_0$
2. Sample random timestep $t \sim \text{Uniform}(1, T)$
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
4. Create noisy image $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$
5. Predict noise $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
6. Minimize $\| \epsilon - \hat{\epsilon} \|^2$

Beautifully simple!

## 8. The U-Net Architecture

Diffusion models typically use a **U-Net** architecture:
- **Encoder**: Downsample image (extract features)
- **Bottleneck**: Process at lowest resolution
- **Decoder**: Upsample back to original size
- **Skip connections**: Preserve spatial information

**Key additions for diffusion:**
1. **Time embedding**: Encode timestep $t$ and inject into the network
2. **Residual blocks**: Better gradient flow
3. **Attention layers**: Capture long-range dependencies (for larger images)

For MNIST, we'll use a simplified U-Net.

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    """
    Sinusoidal embeddings for timesteps (like in Transformers).
    Encodes timestep as a vector that the network can learn from.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

# Test the time embedding
time_emb = SinusoidalPositionEmbeddings(dim=128)
test_times = torch.tensor([0, 100, 500, 999])
test_emb = time_emb(test_times)
print(f"Time embedding shape: {test_emb.shape}")
print(f"Embeddings are unique for each timestep")

In [ ]:
class ResidualBlock(nn.Module):
    """
    Residual block with time embedding injection.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim):
        super().__init__()
        
        # First conv
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        # Time embedding projection
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, out_channels),
            nn.ReLU(inplace=True)
        )
        
        # Second conv
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        # Residual connection (adjust channels if needed)
        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_conv = nn.Identity()
    
    def forward(self, x, t_emb):
        # First conv
        h = self.conv1(x)
        
        # Add time embedding (broadcast across spatial dimensions)
        time_emb = self.time_mlp(t_emb)
        h = h + time_emb[:, :, None, None]
        
        # Second conv
        h = self.conv2(h)
        
        # Residual connection
        return h + self.residual_conv(x)

In [ ]:
class SimpleUNet(nn.Module):
    """
    Simplified U-Net for diffusion models.
    
    Architecture:
    - Encoder: Downsample with residual blocks
    - Bottleneck: Process at lowest resolution
    - Decoder: Upsample with residual blocks + skip connections
    - Time embedding injected at each residual block
    """
    def __init__(self, in_channels=1, out_channels=1, time_emb_dim=128):
        super().__init__()
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        
        # Encoder (downsampling)
        self.conv_in = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        
        self.down1 = ResidualBlock(64, 64, time_emb_dim)
        self.pool1 = nn.MaxPool2d(2)  # 28x28 -> 14x14
        
        self.down2 = ResidualBlock(64, 128, time_emb_dim)
        self.pool2 = nn.MaxPool2d(2)  # 14x14 -> 7x7
        
        # Bottleneck
        self.bottleneck = ResidualBlock(128, 128, time_emb_dim)
        
        # Decoder (upsampling)
        self.up1 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)  # 7x7 -> 14x14
        self.up_res1 = ResidualBlock(128 + 128, 64, time_emb_dim)  # +128 from skip
        
        self.up2 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)  # 14x14 -> 28x28
        self.up_res2 = ResidualBlock(64 + 64, 64, time_emb_dim)  # +64 from skip
        
        # Output
        self.conv_out = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x, t):
        """
        Args:
            x: Input image [B, C, H, W]
            t: Timestep [B]
        
        Returns:
            Predicted noise [B, C, H, W]
        """
        # Encode time
        t_emb = self.time_mlp(t)
        
        # Initial conv
        x = self.conv_in(x)
        
        # Encoder
        skip1 = self.down1(x, t_emb)
        x = self.pool1(skip1)
        
        skip2 = self.down2(x, t_emb)
        x = self.pool2(skip2)
        
        # Bottleneck
        x = self.bottleneck(x, t_emb)
        
        # Decoder with skip connections
        x = self.up1(x)
        x = torch.cat([x, skip2], dim=1)  # Concatenate skip connection
        x = self.up_res1(x, t_emb)
        
        x = self.up2(x)
        x = torch.cat([x, skip1], dim=1)  # Concatenate skip connection
        x = self.up_res2(x, t_emb)
        
        # Output
        x = self.conv_out(x)
        
        return x

# Create model
model = SimpleUNet(in_channels=1, out_channels=1, time_emb_dim=128).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# Test forward pass
test_img = torch.randn(4, 1, 28, 28).to(device)
test_t = torch.randint(0, 1000, (4,)).to(device)
test_output = model(test_img, test_t)
print(f"Input shape: {test_img.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Model successfully predicts noise!")

## 9. Training the Diffusion Model

Now we implement the training loop following the DDPM algorithm.

In [ ]:
def diffusion_loss(model, x0, t, noise, sqrt_alpha_bar, sqrt_one_minus_alpha_bar):
    """
    Compute the diffusion training loss.
    
    Args:
        model: Noise prediction model
        x0: Clean images [B, C, H, W]
        t: Timesteps [B]
        noise: Ground truth noise [B, C, H, W]
        sqrt_alpha_bar: Precomputed sqrt(α̅_t)
        sqrt_one_minus_alpha_bar: Precomputed sqrt(1 - α̅_t)
    
    Returns:
        loss: MSE between true noise and predicted noise
    """
    # Create noisy images
    sqrt_alpha_bar_t = sqrt_alpha_bar[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alpha_bar_t = sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
    
    xt = sqrt_alpha_bar_t * x0 + sqrt_one_minus_alpha_bar_t * noise
    
    # Predict noise
    predicted_noise = model(xt, t)
    
    # MSE loss
    loss = F.mse_loss(predicted_noise, noise)
    
    return loss

In [ ]:
# Training setup
num_epochs = 10
learning_rate = 2e-4

# Reinitialize model and optimizer
model = SimpleUNet(in_channels=1, out_channels=1, time_emb_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Move diffusion parameters to device
sqrt_alphas_cumprod = sqrt_alphas_cumprod.to(device)
sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod.to(device)

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Diffusion steps: {T}")

In [ ]:
# Training loop
losses = []

print("Training diffusion model...\n")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch_idx, (images, _) in enumerate(progress_bar):
        images = images.to(device)
        batch_size_actual = images.size(0)
        
        # Sample random timesteps for each image in batch
        t = torch.randint(0, T, (batch_size_actual,), device=device).long()
        
        # Sample noise
        noise = torch.randn_like(images)
        
        # Compute loss
        loss = diffusion_loss(
            model, images, t, noise,
            sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod
        )
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track loss
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Average loss for epoch
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Diffusion Model Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Sampling: Generating Images

Now the exciting part - generating images from pure noise!

### The Reverse Diffusion Sampling Algorithm

Starting from $x_T \sim \mathcal{N}(0, I)$, iteratively denoise:

For $t = T, T-1, \ldots, 1$:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$

Where:
- $\epsilon_\theta(x_t, t)$ is our trained model's noise prediction
- $\sigma_t$ is the noise variance at step $t$
- $z \sim \mathcal{N}(0, I)$ is added noise (for stochasticity)

**Intuition**: 
1. Model predicts what noise is in $x_t$
2. Remove that predicted noise
3. Add a small amount of fresh noise (except at final step)
4. Repeat until we have a clean image

In [ ]:
@torch.no_grad()
def sample_ddpm(model, n_samples, img_size=(1, 28, 28), device='cpu'):
    """
    Generate samples using DDPM sampling.
    
    Args:
        model: Trained diffusion model
        n_samples: Number of images to generate
        img_size: Image dimensions (C, H, W)
        device: Device to use
    
    Returns:
        Generated images [n_samples, C, H, W]
    """
    model.eval()
    
    # Start from pure noise
    x = torch.randn(n_samples, *img_size).to(device)
    
    # Precompute coefficients
    betas_device = betas.to(device)
    alphas_device = alphas.to(device)
    alphas_cumprod_device = alphas_cumprod.to(device)
    
    # Iteratively denoise
    for t in tqdm(reversed(range(T)), total=T, desc="Sampling"):
        # Current timestep for all samples
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(x, t_batch)
        
        # Get coefficients
        alpha_t = alphas_device[t]
        alpha_bar_t = alphas_cumprod_device[t]
        beta_t = betas_device[t]
        
        # Compute mean of x_{t-1}
        mean = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise
        )
        
        if t > 0:
            # Add noise (except at final step)
            noise = torch.randn_like(x)
            
            # Compute variance (using the fixed variance from DDPM paper)
            alpha_bar_t_prev = alphas_cumprod_device[t-1] if t > 0 else torch.tensor(1.0).to(device)
            variance = beta_t * (1 - alpha_bar_t_prev) / (1 - alpha_bar_t)
            sigma = torch.sqrt(variance)
            
            x = mean + sigma * noise
        else:
            # Final step: no noise
            x = mean
    
    return x

In [ ]:
# Generate samples!
print("Generating images from pure noise...\n")
generated_images = sample_ddpm(model, n_samples=64, img_size=(1, 28, 28), device=device)

# Display
show_images(generated_images, "Generated Images from Diffusion Model")

### Observations

Look at the generated digits:
- Are they recognizable as MNIST digits?
- Is there variety in the samples?
- Do they look sharper than VAE outputs?

With more training epochs (20-30), quality improves significantly!

## 11. Visualizing the Denoising Process

Let's watch the reverse diffusion process in action - noise gradually becoming an image.

In [ ]:
@torch.no_grad()
def sample_with_trajectory(model, img_size=(1, 28, 28), device='cpu', save_every=100):
    """
    Generate a single sample and save intermediate steps.
    """
    model.eval()
    
    # Start from pure noise
    x = torch.randn(1, *img_size).to(device)
    
    trajectory = [x.clone()]
    timesteps_saved = [T]
    
    # Precompute coefficients
    betas_device = betas.to(device)
    alphas_device = alphas.to(device)
    alphas_cumprod_device = alphas_cumprod.to(device)
    
    for t in tqdm(reversed(range(T)), total=T, desc="Sampling trajectory"):
        t_batch = torch.tensor([t], device=device, dtype=torch.long)
        
        predicted_noise = model(x, t_batch)
        
        alpha_t = alphas_device[t]
        alpha_bar_t = alphas_cumprod_device[t]
        beta_t = betas_device[t]
        
        mean = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise
        )
        
        if t > 0:
            noise = torch.randn_like(x)
            alpha_bar_t_prev = alphas_cumprod_device[t-1]
            variance = beta_t * (1 - alpha_bar_t_prev) / (1 - alpha_bar_t)
            sigma = torch.sqrt(variance)
            x = mean + sigma * noise
        else:
            x = mean
        
        # Save intermediate steps
        if t % save_every == 0 or t == 0:
            trajectory.append(x.clone())
            timesteps_saved.append(t)
    
    return torch.cat(trajectory, dim=0), timesteps_saved

# Generate trajectory
trajectory, timesteps = sample_with_trajectory(model, img_size=(1, 28, 28), device=device, save_every=100)

# Display
grid = make_grid(trajectory, nrow=11, padding=2, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(16, 3))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
plt.title(f"Reverse Diffusion Process: t = {timesteps[::-1]}")
plt.axis('off')
plt.tight_layout()
plt.show()

print("Watch as pure noise gradually becomes a digit!")

### Key Insight

Notice the progression:
1. **Early steps (high t)**: Barely visible changes, still looks like noise
2. **Middle steps**: Rough shape emerges
3. **Late steps (low t)**: Fine details are refined

This is a **coarse-to-fine** generation process!

## 12. Faster Sampling: DDIM

**Problem**: DDPM requires 1000 steps - very slow!

**Solution**: DDIM (Denoising Diffusion Implicit Models) - skip steps!

### The Idea

Instead of going $T \rightarrow T-1 \rightarrow T-2 \rightarrow \cdots \rightarrow 0$, 
we can jump: $T \rightarrow 800 \rightarrow 600 \rightarrow \cdots \rightarrow 0$

**DDIM formula** (deterministic version):

$$x_{t-\Delta t} = \sqrt{\bar{\alpha}_{t-\Delta t}} \left( \frac{x_t - \sqrt{1-\bar{\alpha}_t} \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}} \right) + \sqrt{1-\bar{\alpha}_{t-\Delta t}} \epsilon_\theta(x_t, t)$$

**Key benefit**: Can generate in 50 steps instead of 1000 (20x faster!)

In [ ]:
@torch.no_grad()
def sample_ddim(model, n_samples, img_size=(1, 28, 28), device='cpu', num_inference_steps=50):
    """
    Generate samples using DDIM (faster sampling).
    
    Args:
        model: Trained diffusion model
        n_samples: Number of images to generate
        img_size: Image dimensions
        device: Device to use
        num_inference_steps: Number of denoising steps (< T for speedup)
    """
    model.eval()
    
    # Create timestep schedule (evenly spaced subset)
    step_size = T // num_inference_steps
    timesteps = list(range(0, T, step_size))[::-1]  # Reverse order
    
    # Start from pure noise
    x = torch.randn(n_samples, *img_size).to(device)
    
    alphas_cumprod_device = alphas_cumprod.to(device)
    
    for i, t in enumerate(tqdm(timesteps, desc="DDIM Sampling")):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(x, t_batch)
        
        # Get alpha values
        alpha_bar_t = alphas_cumprod_device[t]
        
        if i < len(timesteps) - 1:
            # Next timestep
            t_prev = timesteps[i + 1]
            alpha_bar_t_prev = alphas_cumprod_device[t_prev]
        else:
            # Final step
            alpha_bar_t_prev = torch.tensor(1.0).to(device)
        
        # Predict x0 from xt
        pred_x0 = (x - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
        
        # Clip to valid range
        pred_x0 = torch.clamp(pred_x0, -1, 1)
        
        # Compute x_{t-1}
        if i < len(timesteps) - 1:
            x = torch.sqrt(alpha_bar_t_prev) * pred_x0 + \
                torch.sqrt(1 - alpha_bar_t_prev) * predicted_noise
        else:
            x = pred_x0
    
    return x

# Compare DDIM vs DDPM speed and quality
print("Generating with DDIM (50 steps)...")
import time
start = time.time()
ddim_samples = sample_ddim(model, n_samples=64, img_size=(1, 28, 28), device=device, num_inference_steps=50)
ddim_time = time.time() - start

print(f"DDIM generation time: {ddim_time:.2f}s")
show_images(ddim_samples, "DDIM Generated Images (50 steps)")

### DDPM vs DDIM Trade-off

- **DDPM**: Slower (1000 steps), stochastic, slightly better diversity
- **DDIM**: Faster (50 steps), deterministic, similar quality

For most applications, DDIM is preferred for its speed!

## 13. Comparing Generative Models

Let's compare diffusion models to VAE and GAN (conceptually).

### Quality Comparison

| Model | Quality | Training | Sampling Speed | Diversity |
|-------|---------|----------|----------------|----------|
| **VAE** | Blurry | Stable | Fast (1 step) | Good |
| **GAN** | Sharp | Unstable | Fast (1 step) | Mode collapse risk |
| **Diffusion** | Sharp | Stable | Slow (50-1000 steps) | Excellent |

### When to Use Each?

**VAE:**
- Need fast sampling
- Want interpretable latent space
- Need likelihood estimates
- Compression applications

**GAN:**
- Need fastest sampling
- Maximum image quality priority
- Can handle training instability
- Real-time applications

**Diffusion:**
- Best image quality needed
- Stable training important
- Can afford slower sampling
- Current SOTA for most tasks

### Why Diffusion Wins Today

1. **Best of both worlds**: VAE stability + GAN quality
2. **Scalability**: Works well from small to huge models
3. **Flexibility**: Easy to condition (text, class, etc.)
4. **No mode collapse**: Excellent diversity

## 14. Conditional Diffusion Models

We can extend diffusion models to be **conditional** - control what we generate!

### Class-Conditional Generation

Modify the model to take class label $y$ as input:

$$\epsilon_\theta(x_t, t, y)$$

**Implementation**: Embed class label and add to time embedding.

### Text-Conditional Generation (Stable Diffusion)

Use a text encoder (like CLIP) to embed text prompts:

$$\epsilon_\theta(x_t, t, \text{text\_embed})$$

This is how Stable Diffusion works!

### Classifier-Free Guidance

**Problem**: How to make conditioning strong?

**Solution**: Train model both conditionally and unconditionally, then interpolate:

$$\tilde{\epsilon}_\theta(x_t, t, y) = \epsilon_\theta(x_t, t, \emptyset) + w \cdot (\epsilon_\theta(x_t, t, y) - \epsilon_\theta(x_t, t, \emptyset))$$

Where $w$ is the guidance scale:
- $w = 0$: Unconditional (ignore class)
- $w = 1$: Normal conditional
- $w > 1$: Strong conditioning (exaggerate class features)

This is why you can control "creativity" in Stable Diffusion!

## 15. Advanced Topics

### Latent Diffusion Models (LDM)

**Problem**: Running diffusion on high-res images is expensive!

**Solution**: 
1. Train an autoencoder to compress images to latent space
2. Run diffusion in latent space (much smaller)
3. Decode back to pixel space

This is **Stable Diffusion**! Works on 64×64 latents instead of 512×512 pixels.

**Benefits**:
- 4-8x faster
- Lower memory
- Same quality

### Score-Based Generative Models

Alternative formulation: Instead of predicting noise, predict the **score** (gradient of log probability):

$$s_\theta(x_t, t) = \nabla_{x_t} \log p(x_t)$$

Mathematically equivalent to DDPM, but different training objective.

### Continuous Time Diffusion

Instead of discrete timesteps, use continuous time:

$$dx = f(x, t)dt + g(t)dw$$

This is a **stochastic differential equation (SDE)**. Enables:
- Flexible sampling schedules
- ODE solvers for deterministic sampling
- Theoretical analysis

### Cascaded Diffusion

Generate high-resolution images in stages:
1. Generate 64×64 image
2. Upsample to 256×256 with super-resolution diffusion
3. Upsample to 1024×1024 with another diffusion

Used in DALL-E 2 and Imagen.

## 16. Practical Tips for Training Diffusion Models

### Data Preprocessing
- Normalize to [-1, 1] (works better than [0, 1])
- Augmentation helps (especially for small datasets)
- Resolution matters - start small, scale up

### Model Architecture
- U-Net is standard, attention helps for high-res
- Time embedding is critical - don't skip it!
- Residual connections improve gradient flow
- GroupNorm often better than BatchNorm for small batches

### Training
- Adam optimizer, lr ~ 1e-4 to 2e-4
- EMA (exponential moving average) of weights improves quality
- Gradient clipping helps stability
- Training is expensive - expect 100k+ steps

### Noise Schedule
- Linear works for simple images (MNIST, CIFAR-10)
- Cosine better for complex images (ImageNet, faces)
- Experiment with T (500-1000 typical)

### Sampling
- Use DDIM for speed during development
- DDPM for final high-quality samples
- Fewer steps = faster but lower quality
- Can interpolate timesteps for very fast sampling

### Debugging
- Check if model predicts noise correctly (loss should decrease)
- Visualize samples early - should improve gradually
- Monitor loss across different timesteps
- If samples are blurry, train longer or increase model capacity

## 17. Summary and Key Takeaways

### Core Concepts

1. **Forward Process**: Gradually add noise to images over T steps
   - Can jump directly to any timestep using closed form
   - Controlled by noise schedule β_t

2. **Reverse Process**: Learn to denoise iteratively
   - Train model to predict noise at each timestep
   - Use U-Net with time embeddings
   - Simple training objective: MSE between true and predicted noise

3. **Sampling**: Start from noise, denoise step by step
   - DDPM: 1000 steps, stochastic
   - DDIM: 50 steps, deterministic, faster

### Why Diffusion Models Are Powerful

- **State-of-the-art quality**: Best image generation today
- **Stable training**: No adversarial dynamics like GANs
- **Excellent diversity**: No mode collapse issues
- **Flexible conditioning**: Easy to add text, class, etc.
- **Theoretical foundation**: Strong probabilistic framework

### The Trade-off

- **Slow sampling**: Need many denoising steps
- **Computational cost**: Training is expensive
- **Memory intensive**: Especially for high-resolution

### Real-World Impact

Diffusion models power:
- **DALL-E 2, 3**: Text-to-image generation
- **Stable Diffusion**: Open-source text-to-image
- **Midjourney**: Artistic image generation
- **Imagen**: Google's text-to-image
- **Make-A-Video**: Text-to-video

### Next Steps

1. **Experiment**: Try different noise schedules, architectures
2. **Scale up**: Train on CIFAR-10 or CelebA
3. **Add conditioning**: Implement class-conditional generation
4. **Try latent diffusion**: Combine with VAE for efficiency
5. **Explore variants**: Score-based models, cascaded diffusion

### Further Reading

**Papers:**
- DDPM: "Denoising Diffusion Probabilistic Models" (Ho et al., 2020)
- DDIM: "Denoising Diffusion Implicit Models" (Song et al., 2020)
- Stable Diffusion: "High-Resolution Image Synthesis with Latent Diffusion Models" (Rombach et al., 2022)
- Classifier-Free Guidance: "Classifier-Free Diffusion Guidance" (Ho & Salimans, 2022)

**Resources:**
- Hugging Face Diffusers library
- Lilian Weng's blog on diffusion models
- "What are Diffusion Models?" by Lil'Log

## 18. Reflection Questions

1. **Conceptual Understanding**:
   - Can you explain the forward and reverse diffusion processes in your own words?
   - Why do we predict noise instead of the denoised image directly?
   - How does the noise schedule affect generation quality?

2. **Comparison with Other Models**:
   - What are the key advantages of diffusion models over VAEs?
   - What are the key advantages over GANs?
   - In what scenarios would you still choose VAE or GAN?

3. **Practical Application**:
   - How would you modify this code to generate CIFAR-10 images?
   - What changes would you make for class-conditional generation?
   - How could you reduce sampling time while maintaining quality?

4. **Advanced Thinking**:
   - Why does latent diffusion (Stable Diffusion) work?
   - How does classifier-free guidance give us control over generation?
   - What are the ethical implications of powerful generative models?

Take time to think through these - understanding the "why" is more valuable than memorizing the "how"!